# NB00 — Database Setup (Schema v2.2)
**TCC Weathering ML — Self-contained schema**

Creates `weathering.db` with all tables, indices, and initial data
in a single notebook — no external `.sql` file dependencies.

**Schema v2.2** — the shipped `weathering.db` has 44 tables, 0 views, 15 indices

| Layer | Tables | Content | Populated by |
|---|---|---|---|
| **1** | 5 | Metadata: oils, compounds, stages, model_configs, db_metadata | NB00, NB01 |
| **2** | 6 | Raw: measurements, kinetics, physical properties, pan evap, evap eq | NB01, NB03 |
| **3** | 4 | Derived: ratios, deltas, consistency, correlation filter | NB02, NB03 |
| **4a** | 7 | ML results: LOOO, SHAP, H3, bootstrap, interactions | NB04, NB06 |
| **4b** | 6 | Stability: permutation, subsample, seed, PCHIP, perturbation | _(empty in this deposit)_ |

Layers 1–4 are the core ML schema; the remaining tables hold domain EDA outputs produced by the `03*` notebooks.

**Run ONCE before NB01.**


## 1. Setup

Import `DB_PATH` from the shared `utils.py` module (single source of truth for
all project paths). The database is created in `data/processed/weathering.db`.

This notebook uses `sqlite3.connect()` directly instead of `utils.get_conn()`
because `executescript()` issues implicit commits that are incompatible with
the context manager's rollback logic.


In [ ]:
import sqlite3
import sys
sys.path.insert(0, '.')
from utils import DB_PATH

DB_DIR = DB_PATH.parent
DB_DIR.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists():
    print(f'⚠️  Database already exists: {DB_PATH}')
    print('   All statements use IF NOT EXISTS — safe to re-run.')
else:
    print(f'Creating database: {DB_PATH}')


## 2. Create all tables

Single `executescript()` with the complete schema — no external `.sql` dependencies.
All 29 tables organized in layers:

- **Layer 1 (Metadata):** Core entities. `oils` has 22 columns including physical
  properties (populated by NB01). `compounds` tracks
  all 107 analytes with `excluded` flag for ≥50% missing. `weathering_stages`
  holds 10 stage codes (W0-W4 + biodiesel + Orimulsion variants).
- **Layer 2 (Raw):** Primary data. `measurements` is the core table (~35k rows)
  with raw and imputed values. `sample_properties` is EAV for all physical/chemical
  properties (~50k rows). `pan_evaporation` and `evaporation_equations` hold kinetic data.
- **Layer 3 (Derived):** Computed from Layer 2. Diagnostic ratios (NB02), feature
  deltas W0→W3 (NB03), consistency statistics, and correlation filter log.
- **Layer 4 (ML + Validation):** LOOO metrics/predictions (NB04), SHAP values
  and hierarchy (NB06).

All statements use `CREATE TABLE IF NOT EXISTS` — safe to re-run.


In [ ]:
# Direct connection (not get_conn) — executescript needs implicit commit control
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
PRAGMA journal_mode = WAL;
PRAGMA foreign_keys = ON;

-- =============================================================
-- LAYER 1 — METADATA
-- =============================================================

CREATE TABLE IF NOT EXISTS oils (
    oil_id              INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_name            TEXT    NOT NULL UNIQUE,
    base_name           TEXT    NOT NULL,
    vintage_year        TEXT,
    region              TEXT,
    oil_type            TEXT    NOT NULL
                        CHECK (oil_type IN (
                            'crude', 'refined', 'synthetic', 'bitumen_blend',
                            'biodiesel', 'fuel_oil', 'bitumen', 'orimulsion'
                        )),
    is_multi_vintage    INTEGER NOT NULL DEFAULT 0,
    vintage_group       TEXT,
    include_in_analysis INTEGER NOT NULL DEFAULT 1,
    exclude_reason      TEXT,
    notes               TEXT,
    api_gravity         REAL,
    pour_point_c        REAL,
    density_15c         REAL,
    flash_point_c       REAL,
    vapor_pressure_kpa  REAL,
    sulfur_pct          REAL,
    wax_pct             REAL,
    cluster_gmm         INTEGER,
    source              TEXT,
    date_received       TEXT,
    ests_code           TEXT,
    comments            TEXT
);

CREATE TABLE IF NOT EXISTS weathering_stages (
    stage_code           TEXT    PRIMARY KEY,
    mass_loss_pct_median REAL,
    mass_loss_pct_p25    REAL,
    mass_loss_pct_p75    REAL,
    interpretation       TEXT,
    include_in_analysis  INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE IF NOT EXISTS compounds (
    compound_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    compound_name   TEXT    NOT NULL UNIQUE,
    compound_group  TEXT    NOT NULL
                    CHECK (compound_group IN (
                        'n_alkane', 'isoprenoid',
                        'pah_alkylated', 'pah_priority',
                        'biomarker_terpane', 'biomarker_hopane', 'biomarker_sterane'
                    )),
    carbon_number   INTEGER,
    is_normalizator INTEGER NOT NULL DEFAULT 0,
    missing_pct     REAL,
    excluded        INTEGER NOT NULL DEFAULT 0,
    exclude_reason  TEXT,
    eccc_row_index  INTEGER,
    cen_class       TEXT
);

CREATE TABLE IF NOT EXISTS model_configs (
    config      TEXT PRIMARY KEY,
    description TEXT NOT NULL,
    feature_set TEXT NOT NULL,
    n_features  INTEGER,
    n_oils      INTEGER NOT NULL,
    oil_filter  TEXT,
    algorithm   TEXT NOT NULL DEFAULT 'xgboost',
    purpose     TEXT,
    hypothesis  TEXT
);

CREATE TABLE IF NOT EXISTS db_metadata (
    key         TEXT PRIMARY KEY,
    value       TEXT NOT NULL,
    updated_at  TEXT DEFAULT (datetime('now'))
);

-- =============================================================
-- LAYER 2 — RAW DATA
-- =============================================================

CREATE TABLE IF NOT EXISTS measurements (
    measurement_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL REFERENCES weathering_stages(stage_code) ON DELETE CASCADE,
    compound_id     INTEGER NOT NULL REFERENCES compounds(compound_id) ON DELETE CASCADE,
    value_raw       REAL,
    value_imputed   REAL,
    is_nd           INTEGER NOT NULL DEFAULT 0,
    is_imputed      INTEGER NOT NULL DEFAULT 0,
    UNIQUE (oil_id, stage_code, compound_id)
);

CREATE TABLE IF NOT EXISTS oil_weathering_kinetics (
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL,
    time_hours      REAL,
    mass_loss_pct   REAL,
    method          TEXT    DEFAULT 'pan_evaporation_15C',
    PRIMARY KEY (oil_id, stage_code)
);

CREATE TABLE IF NOT EXISTS augmented_measurements (
    aug_id              INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_id              INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_continuous    REAL    NOT NULL,
    stage_code_nearest  TEXT,
    compound_id         INTEGER NOT NULL REFERENCES compounds(compound_id) ON DELETE CASCADE,
    value_interpolated  REAL,
    method              TEXT    DEFAULT 'pchip',
    is_original         INTEGER NOT NULL,
    UNIQUE (oil_id, stage_continuous, compound_id)
);

CREATE TABLE IF NOT EXISTS sample_properties (
    prop_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL,
    property_name   TEXT    NOT NULL,
    temperature_c   REAL,
    condition       TEXT,
    value           REAL,
    unit            TEXT,
    std_dev         REAL,
    replicates      INTEGER,
    method          TEXT,
    UNIQUE (oil_id, stage_code, property_name, temperature_c, condition)
);

CREATE TABLE IF NOT EXISTS pan_evaporation (
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    time_hours      REAL    NOT NULL,
    mass_loss_pct   REAL,
    temperature_c   REAL    DEFAULT 15.0,
    PRIMARY KEY (oil_id, time_hours)
);

CREATE TABLE IF NOT EXISTS evaporation_equations (
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    equation_type   TEXT    NOT NULL
                    CHECK (equation_type IN ('ln', 'sqrt', 'ln_C')),
    coeff_a         REAL,
    coeff_b         REAL,
    coeff_c         REAL,
    PRIMARY KEY (oil_id, equation_type)
);

-- =============================================================
-- LAYER 3 — DERIVED
-- =============================================================

CREATE TABLE IF NOT EXISTS diagnostic_ratios (
    ratio_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL REFERENCES weathering_stages(stage_code) ON DELETE CASCADE,
    ratio_name      TEXT    NOT NULL,
    ratio_type      TEXT    NOT NULL
                    CHECK (ratio_type IN ('process', 'identity', 'depletion', 'source', 'exploratory')),
    numerator       TEXT    NOT NULL,
    denominator     TEXT    NOT NULL,
    value           REAL,
    is_valid        INTEGER NOT NULL DEFAULT 1,
    cen_category    TEXT,
    UNIQUE (oil_id, stage_code, ratio_name)
);

CREATE TABLE IF NOT EXISTS ratio_definitions (
    ratio_name      TEXT    PRIMARY KEY,
    ratio_type      TEXT    NOT NULL
        CHECK (ratio_type IN ('process','identity','depletion','source','exploratory')),
    numerator       TEXT    NOT NULL,
    denominator     TEXT    NOT NULL,
    category        TEXT    NOT NULL DEFAULT 'canonical'
        CHECK (category IN ('canonical','exploratory','derived','alternative')),
    is_sum_ratio    INTEGER NOT NULL DEFAULT 0,
    cen_reference   TEXT,
    literature_ref  TEXT,
    notes           TEXT
);

-- =============================================================
-- DEFENSE-IN-DEPTH TRIGGERS (CHG-0004, 24/abr/2026)
-- Reverse-propagation from NB02 audit: ratio_definitions.numerator/denominator
-- are TEXT free-form but must match compounds.compound_name for ratio
-- computation in NB02 to succeed. Without triggers, encoding drift (e.g.
-- F-NB01-C1 ß→ss transliteration) silently produces "defined-not-populated"
-- ratios. These triggers fail-fast at INSERT time.
-- Pseudo-names ('sum:*', 'ratio:*') are exempted — they aggregate compounds
-- and are not validated against the compounds table.
-- =============================================================

DROP TRIGGER IF EXISTS ratio_defs_validate_numerator;
DROP TRIGGER IF EXISTS ratio_defs_validate_denominator;

CREATE TRIGGER ratio_defs_validate_numerator
AFTER INSERT ON ratio_definitions
WHEN NEW.numerator NOT LIKE 'sum:%' AND NEW.numerator NOT LIKE 'ratio:%'
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (SELECT 1 FROM compounds WHERE compound_name = NEW.numerator)
        THEN RAISE(ABORT, 'ratio_definitions.numerator not found in compounds table (check NB01 name normalization; see F-NB01-C1, CHG-0003)')
    END;
END;

CREATE TRIGGER ratio_defs_validate_denominator
AFTER INSERT ON ratio_definitions
WHEN NEW.denominator NOT LIKE 'sum:%' AND NEW.denominator NOT LIKE 'ratio:%'
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (SELECT 1 FROM compounds WHERE compound_name = NEW.denominator)
        THEN RAISE(ABORT, 'ratio_definitions.denominator not found in compounds table (check NB01 name normalization; see F-NB01-C1, CHG-0003)')
    END;
END;

CREATE TABLE IF NOT EXISTS feature_deltas (
    delta_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    feature_name    TEXT    NOT NULL,
    feature_type    TEXT    NOT NULL
                    CHECK (feature_type IN ('compound', 'ratio')),
    value_w0        REAL,
    value_w3        REAL,
    delta_pct       REAL,
    delta_abs       REAL,
    UNIQUE (oil_id, feature_name)
);

CREATE TABLE IF NOT EXISTS feature_consistency (
    feature_name    TEXT    PRIMARY KEY,
    feature_type    TEXT    NOT NULL
                    CHECK (feature_type IN ('compound', 'ratio')),
    delta_mean      REAL,
    delta_median    REAL,
    delta_std       REAL,
    cv_pct          REAL,
    n_oils          INTEGER,
    lability_rank   INTEGER
);

CREATE TABLE IF NOT EXISTS correlation_filter_log (
    pair_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    threshold       REAL    NOT NULL,
    removed_feature TEXT    NOT NULL,
    kept_feature    TEXT    NOT NULL,
    pearson_r       REAL    NOT NULL,
    cv_removed      REAL,
    cv_kept         REAL,
    removal_order   INTEGER
);

-- =============================================================
-- LAYER 4a — ML RESULTS
-- =============================================================

CREATE TABLE IF NOT EXISTS looo_metrics (
    fold_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    config          TEXT    NOT NULL,
    test_oil_id     INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    mae             REAL,
    rmse            REAL,
    r_squared       REAL,
    pct_within_1    REAL,
    n_test_samples  INTEGER,
    model_params    TEXT,
    run_id          TEXT,
    UNIQUE (config, test_oil_id)
);

CREATE TABLE IF NOT EXISTS looo_predictions (
    pred_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    fold_id         INTEGER NOT NULL REFERENCES looo_metrics(fold_id) ON DELETE CASCADE,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL REFERENCES weathering_stages(stage_code) ON DELETE CASCADE,
    y_true          REAL,
    y_pred          REAL,
    abs_error       REAL
);

CREATE TABLE IF NOT EXISTS shap_values (
    shap_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    fold_id         INTEGER NOT NULL REFERENCES looo_metrics(fold_id) ON DELETE CASCADE,
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_code      TEXT    NOT NULL REFERENCES weathering_stages(stage_code) ON DELETE CASCADE,
    feature_name    TEXT    NOT NULL,
    shap_value      REAL    NOT NULL,
    feature_value   REAL
);

CREATE TABLE IF NOT EXISTS shap_hierarchy (
    hierarchy_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    config          TEXT    NOT NULL DEFAULT 'C1',
    stage_code      TEXT    NOT NULL,
    feature_name    TEXT    NOT NULL,
    feature_type    TEXT    NOT NULL
                    CHECK (feature_type IN ('compound', 'ratio')),
    shap_mean_abs   REAL,
    shap_std        REAL,
    cv_shap         REAL,
    importance_rank INTEGER,
    is_robust       INTEGER NOT NULL DEFAULT 0,
    UNIQUE (config, stage_code, feature_name)
);

CREATE TABLE IF NOT EXISTS h3_feature_selection (
    selection_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    n_features      INTEGER NOT NULL,
    stage_filter    TEXT,
    mae_mean        REAL,
    mae_std         REAL,
    pct_within_1    REAL,
    delta_mae_vs_full REAL,
    features_used   TEXT
);

CREATE TABLE IF NOT EXISTS bootstrap_ci (
    config      TEXT    NOT NULL,
    metric      TEXT    NOT NULL,
    point_est   REAL    NOT NULL,
    ci_lower    REAL    NOT NULL,
    ci_upper    REAL    NOT NULL,
    n_bootstrap INTEGER DEFAULT 10000,
    alpha       REAL    DEFAULT 0.05,
    PRIMARY KEY (config, metric)
);

CREATE TABLE IF NOT EXISTS shap_interaction_values (
    config          TEXT    NOT NULL,
    feature_a       TEXT    NOT NULL,
    feature_b       TEXT    NOT NULL,
    interaction_mean REAL,
    interaction_std  REAL,
    n_samples       INTEGER,
    PRIMARY KEY (config, feature_a, feature_b)
);

-- =============================================================
-- LAYER 4b — STABILITY AND SENSITIVITY
-- =============================================================

CREATE TABLE IF NOT EXISTS permutation_importance (
    perm_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    config          TEXT    NOT NULL,
    feature_name    TEXT    NOT NULL,
    importance_mean REAL,
    importance_std  REAL,
    n_repeats       INTEGER DEFAULT 10,
    UNIQUE (config, feature_name)
);

CREATE TABLE IF NOT EXISTS subsample_stability (
    n_oils_used     INTEGER NOT NULL,
    config          TEXT    NOT NULL,
    iteration       INTEGER NOT NULL,
    mae             REAL,
    PRIMARY KEY (n_oils_used, config, iteration)
);

CREATE TABLE IF NOT EXISTS seed_sensitivity (
    seed_value      INTEGER NOT NULL,
    config          TEXT    NOT NULL,
    mae             REAL,
    pct_within_1    REAL,
    PRIMARY KEY (seed_value, config)
);

CREATE TABLE IF NOT EXISTS pchip_validation (
    oil_id          INTEGER NOT NULL REFERENCES oils(oil_id) ON DELETE CASCADE,
    stage_continuous REAL   NOT NULL,
    compound_id     INTEGER NOT NULL REFERENCES compounds(compound_id) ON DELETE CASCADE,
    value_true      REAL,
    value_pchip     REAL,
    abs_error       REAL,
    PRIMARY KEY (oil_id, stage_continuous, compound_id)
);

CREATE TABLE IF NOT EXISTS augmented_model_results (
    config          TEXT    NOT NULL,
    n_augmented     INTEGER NOT NULL,
    mae             REAL,
    pct_within_1    REAL,
    method          TEXT    DEFAULT 'pchip',
    PRIMARY KEY (config, n_augmented)
);

CREATE TABLE IF NOT EXISTS shap_perturbation (
    config          TEXT    NOT NULL,
    perturbation_type TEXT  NOT NULL,
    feature_name    TEXT    NOT NULL,
    shap_mean_abs_original REAL,
    shap_mean_abs_perturbed REAL,
    delta_pct       REAL,
    PRIMARY KEY (config, perturbation_type, feature_name)
);

""");
conn.close()
print(f'✓ All 29 tables created: {DB_PATH}')


## 3. Initial data: weathering stages

Ten stage codes covering: evaporative weathering (W0-W4), biodiesel blends
(B5/B20/B100), and Orimulsion water correction (BWC/AWC).

W0 has mass_loss = 0 (fresh oil). W1-W4 start as NULL — NB01 computes the
empirical median/p25/p75 from actual sample data. This avoids hardcoding
stale statistics from previous pipeline versions.

Only W0-W3 have `include_in_analysis=1`. W4 and non-petroleum stages are excluded.


In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.executescript('''
    INSERT OR IGNORE INTO weathering_stages VALUES
        ('W0',   0.0,  0.0,  0.0,  'Fresh oil — reference',                1),
        ('W1',   NULL, NULL, NULL,  'Light evaporation — stats from NB01',  1),
        ('W2',   NULL, NULL, NULL,  'Moderate evaporation — stats from NB01', 1),
        ('W3',   NULL, NULL, NULL,  'Severe evaporation — stats from NB01', 1),
        ('W4',   NULL, NULL, NULL,  'Extreme evaporation — EXCLUDED',       0),
        ('B5',   NULL, NULL, NULL,  'Biodiesel blend 5%',                   0),
        ('B20',  NULL, NULL, NULL,  'Biodiesel blend 20%',                  0),
        ('B100', NULL, NULL, NULL,  'Biodiesel pure',                       0),
        ('BWC',  NULL, NULL, NULL,  'Before water correction (Orimulsion)', 0),
        ('AWC',  NULL, NULL, NULL,  'After water correction (Orimulsion)',  0);
''');
conn.close()
print('✓ 10 weathering stages inserted (W1-W3 stats populated by NB01)')


## 4. Model configurations and database metadata

**12 model configurations** test different hypotheses:
- C1 (all 87 features): full model baseline
- C2 (29 ratios): domain knowledge — the operationally relevant model
- C3 (58 compounds): are ratios redundant for XGBoost?
- C4/C4b/C4c (PCA variants): dimensionality reduction baselines (H5)
- C6 (Random Forest): algorithm dependency check
- C7 (median dummy): absolute baseline (MAE=1.000)
- C8 (crude only): primary ranking without oil-type confounding
- C2i (identity ratios): do source-diagnostic ratios carry weathering signal?
- Ridge: linearity check — confirms nonlinear signal requires tree models

**16 metadata entries** make the database self-documenting: schema version,
seed, thresholds, methods, and placeholders for counts populated by NB01.


In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')

configs = [
    ('C1',    'XGB-all (mixed) — XGBoost, all features (C62ALL: 142), mixed oil types (44 oils)',        'all',       87, 45, None,        'xgboost',       'Full model',                               None),
    ('C2',    'XGB-ratios — XGBoost, 29 diagnostic ratios only',                  'ratios',    29, 45, None,        'xgboost',       'Domain knowledge (operational)',             'H5'),
    ('C3',    'XGB-compounds — XGBoost, 58 compounds only (no ratios)',              'compounds', 58, 45, None,        'xgboost',       'Ratios redundant?',                         None),
    ('C4',    'PCA of 92 pre-filter compounds (var>=90%) — exploratory, not run, not in paper',  'pca',       16, 45, None,        'xgboost',       'PCA baseline',                              'H5'),
    ('C6',    'RF-all — Random Forest, all features (C62ALL: 142)',            'all',       87, 45, None,        'random_forest', 'Algorithm dependency',                      None),
    ('C7',    'Dummy (crude) — median-prediction baseline, crude scope (no features)',                 'none',       0, 45, None,        'dummy',         'Baseline',                                  'prerequisite'),
    ('C8',    'XGB-all (crude) — XGBoost, all features (C45CRUDE: 127), crude-only (29 oils)',               'all',       87, 36, 'crude_only','xgboost',       'Primary ranking (no oil-type confounding)', None),
    ('C2i',   'XGB-identity — XGBoost, 13 identity ratios only',                    'identity',  13, 45, None,        'xgboost',       'Identity signal check',                     None),
    ('Ridge', 'Ridge — RidgeCV linear baseline, all features (C62ALL: 142)',         'all',       87, 45, None,        'ridge',         'Linearity check (R06)',                     None),
    ('C4b',   'PCA of 29 ratios (fixed 29 PCs) — exploratory, not run, not in paper',            'pca',       29, 45, None,        'xgboost',       'Dimensionality control',                    'H5'),
    ('C4c',   'PCA of 58 post-filter compounds (var>=90%) — exploratory, not run, not in paper', 'pca',       13, 45, None,        'xgboost',       'PCA matched features',                      'H5'),
    ('C7_mixed', 'Dummy (mixed) — median-prediction baseline, mixed scope (NB04 Sessão C, regularized Sessão P)', 'none', 0, 45, None, 'dummy', 'Mixed-type dummy baseline',                None),
]
conn.executemany('INSERT OR REPLACE INTO model_configs VALUES (?,?,?,?,?,?,?,?,?)', configs)

metadata = [
    ('schema_version',          '2.2'),
    ('pipeline_version',        'TCC-v2.0'),
    ('seed',                    '42'),
    ('n_oils_total',            '120'),
    ('n_oils_included',         'set_by_NB01'),
    ('n_compounds_total',       'set_by_NB01'),
    ('n_compounds_active',      'set_by_NB01'),
    ('n_features_post_filter',  '87'),
    ('correlation_threshold',   '0.95'),
    ('missing_threshold_pct',   '50'),
    ('imputation_method',       'ND_to_zero_D15'),
    ('d14_uniform_zeros',       'value_imputed_to_NULL'),
    ('looo_n_trials_optuna',    '50'),
    ('xgboost_objective',       'reg:absoluteerror'),
    ('shap_strategy',           'test_only_A'),
    ('csv_source',              'ECCC_Petroleum_Products_Database-2021-01-22_EN.csv'),
]
conn.executemany('INSERT OR REPLACE INTO db_metadata VALUES (?, ?, datetime("now"))', 
                 [(k, v) for k, v in metadata])

conn.commit()
conn.close()
print(f'✓ {len(configs)} model configurations inserted')
print(f'✓ {len(metadata)} metadata entries inserted')


## Configuration codes → paper names (crosswalk)

The internal config codes below are **values stored in the database**
(`model_configs.config`, `looo_predictions.config_name`, …) and **model-folder
names** (`models/looo_models/<code>/`), so they are kept verbatim throughout the
pipeline. This legend maps each code to the name used in the paper (Table S2).

| Code | Paper name | Model / feature scope |
|---|---|---|
| `C1` | XGB-all (mixed) | XGBoost, all features, mixed oil types — cohort `C62ALL` (142 features, 44 oils) |
| `C8` | XGB-all (crude) | XGBoost, all features, crude-only — cohort `C45CRUDE` (127 features, 29 oils) |
| `C2` | XGB-ratios | XGBoost, diagnostic ratios only |
| `C3` | XGB-compounds | XGBoost, compounds only |
| `C2i` | XGB-identity | XGBoost, identity-class ratios only |
| `C6` | RF-all | Random Forest, all features |
| `Ridge` | Ridge | RidgeCV linear baseline |
| `C7` | Dummy (crude) | median-prediction baseline, crude scope |
| `C7_mixed` | Dummy (mixed) | median-prediction baseline, mixed scope |
| `C4`, `C4b`, `C4c` | *(not in paper)* | PCA exploratory configs — defined in `model_configs` but **not run** (no predictions/models on disk); excluded from Table S2 |

**Naming caveats.**
- The numerals in `C62ALL` / `C45CRUDE` are **not** counts — the actual counts are 142/127 features and 44/29 oils. Verify by row count, never by the code numerals.
- `W0`–`W3` are the four weathering stages (used as-is in the paper).
- `Sessão …`, `D-…`, `CHG-…`, `§18` are internal process labels (development sessions / decisions / changelog / pre-registration discipline), retained for provenance.


## 5. Performance indices

13 indices optimized for the most common query patterns:
- `measurements` by (oil_id, stage_code) — the primary access pattern for ML
- `diagnostic_ratios` by ratio_name — for H3 feature selection loops
- `shap_values` by (stage_code, feature_name) — for per-stage SHAP analysis
- `sample_properties` by property_name and composite (property, stage, oil) —
  the EAV table needs these for efficient filtering


In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.executescript('''
    CREATE INDEX IF NOT EXISTS idx_measurements_oil_stage ON measurements (oil_id, stage_code);
    CREATE INDEX IF NOT EXISTS idx_measurements_compound ON measurements (compound_id);
    CREATE INDEX IF NOT EXISTS idx_ratios_oil_stage ON diagnostic_ratios (oil_id, stage_code);
    CREATE INDEX IF NOT EXISTS idx_ratios_name ON diagnostic_ratios (ratio_name);
    CREATE INDEX IF NOT EXISTS idx_shap_stage_feature ON shap_values (stage_code, feature_name);
    CREATE INDEX IF NOT EXISTS idx_shap_fold ON shap_values (fold_id);
    CREATE INDEX IF NOT EXISTS idx_fcr_config ON fcr_values (config, stage_code);
    CREATE INDEX IF NOT EXISTS idx_oils_type_active ON oils (oil_type, include_in_analysis);
    CREATE INDEX IF NOT EXISTS idx_ratios_valid ON diagnostic_ratios (is_valid, ratio_name);
    CREATE INDEX IF NOT EXISTS idx_sample_props_oil_stage ON sample_properties (oil_id, stage_code);
    CREATE INDEX IF NOT EXISTS idx_sample_props_name ON sample_properties (property_name);
    CREATE INDEX IF NOT EXISTS idx_sample_props_composite ON sample_properties (property_name, stage_code, oil_id);
    CREATE INDEX IF NOT EXISTS idx_pan_evap_oil ON pan_evaporation (oil_id);
''');
conn.close()
print('✓ 13 indices created')


## Verification

Enumerate all tables and assert expected counts and names by layer.
The schema uses tables only; no views are created.


In [ ]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
objects = conn.execute(
    "SELECT name, type FROM sqlite_master "
    "WHERE type = 'table' AND name != 'sqlite_sequence' ORDER BY name"
).fetchall()
conn.close()

tables = [o[0] for o in objects]

# ── Count verification ──
expected_n_tables = 29
if len(tables) != expected_n_tables:
    print(f'Note: {len(tables)} tables present (base schema = {expected_n_tables})')

# ── Name verification by layer ──
expected_tables = {
    'Layer 1 (Metadata)': [
        'oils', 'weathering_stages', 'compounds', 'model_configs', 'db_metadata'],
    'Layer 2 (Raw)': [
        'measurements', 'oil_weathering_kinetics', 'augmented_measurements',
        'sample_properties', 'pan_evaporation', 'evaporation_equations'],
    'Layer 3 (Derived)': [
        'diagnostic_ratios', 'ratio_definitions', 'feature_deltas',
        'feature_consistency', 'correlation_filter_log'],
    'Layer 4a (ML Results)': [
        'looo_metrics', 'looo_predictions', 'shap_values', 'shap_hierarchy',
        'h3_feature_selection', 'bootstrap_ci', 'shap_interaction_values'],
    'Layer 4b (Stability)': [
        'permutation_importance', 'subsample_stability', 'seed_sensitivity',
        'pchip_validation', 'augmented_model_results', 'shap_perturbation'],
}

all_expected = []
for layer, table_list in expected_tables.items():
    all_expected.extend(table_list)
    found = [t for t in table_list if t in tables]
    missing = [t for t in table_list if t not in tables]
    status = '✓' if not missing else '⚠️'
    print(f'{status} {layer}: {len(found)}/{len(table_list)}')
    for m in missing:
        print(f'    MISSING: {m}')

unexpected = [t for t in tables if t not in all_expected]
if unexpected:
    print(f'⚠️  Unexpected tables: {unexpected}')

assert not [t for t in all_expected if t not in tables], \
    f'Missing tables: {[t for t in all_expected if t not in tables]}'

print(f'\n✓ Schema verified: {len(tables)} tables')
print(f'  Views: will be created at end of pipeline')
print(f'  Database size: {DB_PATH.stat().st_size:,} bytes')


## Export schema as .sql (documentation artifact)

Reads `sqlite_master` and exports as formatted `.sql`.
This is an OUTPUT, not input — never read by the pipeline.
Note: views will appear in the export only after the consolidation
notebook creates them.


In [ ]:
schema_dir = DB_PATH.parent.parent.parent / 'schemas'
schema_dir.mkdir(parents=True, exist_ok=True)
schema_path = schema_dir / 'schema_v2.2.sql'

conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    "SELECT type, name, sql FROM sqlite_master "
    "WHERE sql IS NOT NULL ORDER BY "
    "CASE type WHEN 'table' THEN 1 WHEN 'index' THEN 2 WHEN 'view' THEN 3 END, name"
).fetchall()
conn.close()

with open(schema_path, 'w', encoding='utf-8') as f:
    f.write(f'-- Schema v2.2 — exported from weathering.db by NB00\n')
    f.write(f'-- Generated: {pd.Timestamp.now().isoformat()}\n')
    f.write(f'-- Tables: {sum(1 for t,_,_ in rows if t=="table")}\n')
    f.write(f'-- Views:  {sum(1 for t,_,_ in rows if t=="view")}\n')
    f.write(f'-- Indices: {sum(1 for t,_,_ in rows if t=="index")}\n\n')
    current_type = None
    for obj_type, name, sql in rows:
        if obj_type != current_type:
            f.write(f'-- {"=" * 60}\n')
            f.write(f'-- {obj_type.upper()}S\n')
            f.write(f'-- {"=" * 60}\n\n')
            current_type = obj_type
        f.write(sql + ';\n\n')

print(f'✓ Schema exported: {schema_path}')
print(f'  {schema_path.stat().st_size:,} bytes')


## Output

- `weathering.db` — 44 tables + 15 indices
- `schemas/schema_v2.2.sql` — exported documentation artifact
- 12 model configurations + 16 metadata entries + 10 weathering stages

**Architecture:** NB00 creates tables. Each downstream NB creates its own
tables with `CREATE TABLE IF NOT EXISTS`.

**Next:** NB01 — Extract all data from ECCC ESTS CSV.
